In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss
import numpy as np
import requests 
import json
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)


In [3]:
data = np.load('rag_foundation.npz', allow_pickle=True)
kb_embeddings = data['embeddings'].astype('float32')
knowledge_chunk = data['metadata']

dimension = kb_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(kb_embeddings)

# print(f"Loaded {len(knowledge_chunk)} chunks into the index.")

In [4]:
def bm25_preprocess(text):
    return re.sub(r'[^\w\s]', '', text).lower().split()

In [5]:
corpus = [bm25_preprocess(chunk['embedding_text']) for chunk in knowledge_chunk]
bm25 = BM25Okapi(corpus)

In [6]:
def mean_pooling(model_op, attention_mask):

    token_embeds = model_op[0]
    ip_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeds.size()).float()

    sum_embeddings = torch.sum(token_embeds * ip_mask_expanded, 1)
    sum_mask = torch.clamp(ip_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

def get_query_embedding(question):

    encoded_input = tokenizer([question], padding=True, truncation=True, return_tensors='pt', max_length=256).to(device)

    with torch.no_grad():
        model_output = model(**encoded_input)

    sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
    
    return F.normalize(sentence_embeddings, p=2, dim=1).cpu().numpy().astype('float32')

In [7]:
def retrieve(question, tgt_domain, k_get = 50, k_f = 5):
    #retrieval with domain
    embeds = get_query_embedding(question)
    _, vdb_ind = index.search(embeds,k_get)
    vdb_filt = [idx for idx in vdb_ind[0] if knowledge_chunk[idx]['metadata']['domain']==tgt_domain]

    tok_query = bm25_preprocess(question)
    bm25_scores = bm25.get_scores(tok_query)
    bm25_top = np.argsort(bm25_scores)[::-1][:k_get]
    bm25_filt = [idx for idx in bm25_top if knowledge_chunk[idx]['metadata']['domain'] == tgt_domain]

    hybrid_indices = list(set(vdb_filt) | set(bm25_filt))


    if not hybrid_indices:
        return []

    candidate_chunks = [knowledge_chunk[idx] for idx in hybrid_indices]
    pairs = [[question, c['embedding_text']] for c in candidate_chunks]

    scores = reranker.predict(pairs)
    ranked_results = sorted(zip(candidate_chunks, scores), key=lambda x: x[1], reverse=True)

    return [chunk['text'] for chunk,score in ranked_results[:k_f]]

    


In [8]:
def retrieve_global_context(question, k_get=100, k_f=5):
    #retrieval without domain
    q_embed = get_query_embedding(question)
    _, vdb_indices = index.search(q_embed, k_get)
    

    tok_query = bm25_preprocess(question)
    bm25_scores = bm25.get_scores(tok_query)
    bm25_global_top = np.argsort(bm25_scores)[::-1][:k_get]

    
    hybrid_indices = list(set(vdb_indices[0]) | set(bm25_global_top))
    
    candidate_chunks = [knowledge_chunk[idx] for idx in hybrid_indices]
    pairs = [[question, c['embedding_text']] for c in candidate_chunks]
    scores = reranker.predict(pairs)
    ranked_results = sorted(zip(candidate_chunks, scores), key=lambda x: x[1], reverse=True)

    return [chunk['text'] for chunk, score in ranked_results[:k_f]]

In [9]:
def call_ollama(prompt):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": "llama3",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.2  #less temperature so we keep it precise and to the point, not let it get creative
        }
    }
    
    try:
        response = requests.post(url, json=payload)
        return response.json().get('response', "No response from model.")
    except Exception as e:
        return f"Connection Error: {e}. Is Ollama running?"

In [10]:
def ask_llama(question, domain):

    # chunks = retrieve(question, domain)
    chunks = retrieve_global_context(question)

    if not chunks:
        return {"answer": "I couldn't find any relevant documents in that domain.", "sources": []}

    context = "\n\n".join([f"Source {i+1}\n{text}" for i,text in enumerate(chunks)])

    #You are a helpful assistant for {domain.upper()} department.

    prompt = f"""
    You are a helpful assistant. 
    Answer the user's question ONLY using the provided context. 

    STRICT RULES:
    1. Use ONLY the provided context. If the answer isn't there, say you don't know.
    2. Be concise. Start with the most important information.
    3. Prioritize specific timelines (e.g., "within 10 days"), legal requirements, or form names. Remove sources from the answer.
    4. Do not offer general life advice or information not found in the snippets.
    

    CONTEXT:
    {context}

    USER QUESTION:
    {question}

    HELPFUL ANSWER:
    """

    answer = call_ollama(prompt)
    
    return {
        "answer": answer,
        "sources": chunks
    }

In [13]:
test_q = "different types of computer"
test_domain = "dmv"

result = ask_llama(test_q, test_domain)

print(f"QUESTION: {test_q}\n")
print(f"LLAMA 3 RESPONSE:\n{result['answer']}\n")
print("-" * 30)
print(f"SOURCE SNIPPETS FOUND: {len(result['sources'])}")

QUESTION: different types of computer

LLAMA 3 RESPONSE:
According to the provided context, there are three architectural configurations for parallel computers:

1. Pipeline computers: Perform overlapped computations to exploit temporal parallelism.
2. Array processors: Use multiple synchronized arithmetic logic units to achieve spatial parallelism.
3. Multiprocessor systems: Use a set of interactive processors with shared resources (memory, etc.) to achieve asynchronous parallelism.

Additionally, there are two types of computer organizations:

1. SIMD (Single Instruction Multiple Data): In this organization, there are multiple processing elements supervised by the same control unit. All PEs receive the same instruction broadcast from the control unit but operate on different data sets from distinct data streams.
2. MIMD (Multiple Instruction Multiple Data): Most multiprocessor systems and multiple computer systems can be classified in this category. An intrinsic MIMD computer implies

In [12]:
print(result)

{'answer': "Amdahl's Law states that the performance improvement to be gained from using some faster mode of execution is limited by the fraction of the time the faster mode can be used. In this case, it's 5% (the portion that cannot be parallelized) out of a total of 100%, which would take 20 days to run.", 'sources': ['Parallel Programming   Analogy\n10-02-2026\n3\n\nRealistic Expectations\n• Ex. – Your program takes 20 days to run\n• 95% can be parallelized\n• 5% cannot (serial)\n• What is the fastest this code can run?\n• As many CPU’s as you want!\n1 day!\nAmdahl’s Law\nThe performance improvement to be gained from using some faster mode of \nexecution is limited by the fraction of the time the faster mode can be used\n(5/100)*20', 'The law states that no person shall operate a motor vehicle under the influence of alcohol or drugs while a child who is 15 years of age or younger is a passenger in the vehicle.', "Leandra's Law was signed into law on November 18 , 2009 in honor of Le